In [1]:
import jax
import jax.numpy as jnp
from jax import jit, grad
from functools import partial
import jax.scipy.linalg
import numpy as np
from scipy.optimize import curve_fit

# =============================================================================
# 1. CORE JAX ENGINE (Standard SU(3) Lattice QCD)
# =============================================================================

def get_generators():
    gens = []
    gens.append(jnp.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, -1j, 0], [1j, 0, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[1, 0, 0], [0, -1, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 1], [0, 0, 0], [1, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, -1j], [0, 0, 0], [1j, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 0], [0, 0, 1], [0, 1, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 0], [0, 0, -1j], [0, 1j, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, -2]], dtype=jnp.complex64) * (0.5/jnp.sqrt(3)))
    return jnp.stack(gens)

GENERATORS = get_generators()

@partial(jit, static_argnums=(1, 2, 3))
def wilson_loop_trace(U_field, origin, R, T):
    L = U_field.shape[0]
    x, y, z, t = origin
    prod = jnp.eye(3, dtype=jnp.complex64)
    # +mu
    for r in range(R): prod = prod @ U_field[(x+r) % L, y, z, t, 0]
    # +nu
    for r in range(T): prod = prod @ U_field[(x+R) % L, (y+r) % L, z, t, 1]
    # -mu
    for r in range(R): prod = prod @ jnp.conjugate(jnp.swapaxes(U_field[(x+R-1-r) % L, (y+T) % L, z, t, 0], -1, -2))
    # -nu
    for r in range(T): prod = prod @ jnp.conjugate(jnp.swapaxes(U_field[x, (y+T-1-r) % L, z, t, 1], -1, -2))
    return jnp.real(jnp.trace(prod)) / 3.0

@jit
def wilson_action(U_field, beta):
    total_trace = 0.0
    for mu in range(4):
        for nu in range(mu + 1, 4):
            U_mu = U_field[:, :, :, :, mu]
            U_nu_shift_mu = jnp.roll(U_field[:, :, :, :, nu], -1, axis=mu)
            U_mu_shift_nu = jnp.roll(U_field[:, :, :, :, mu], -1, axis=nu)
            U_nu = U_field[:, :, :, :, nu]
            U_mu_dag = jnp.conjugate(jnp.swapaxes(U_mu_shift_nu, -1, -2))
            U_nu_dag = jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
            P = U_mu @ U_nu_shift_mu @ U_mu_dag @ U_nu_dag
            total_trace += jnp.sum(jnp.real(jnp.trace(P, axis1=-2, axis2=-1)))
    return -(beta / 3.0) * total_trace

@jit
def langevin_step(U_field, key, epsilon, beta):
    dS_dU = grad(wilson_action, argnums=0)(U_field, beta)
    U_dag = jnp.conjugate(jnp.swapaxes(U_field, -1, -2))
    force_raw = U_dag @ dS_dU
    force_ah = (force_raw - jnp.conjugate(jnp.swapaxes(force_raw, -1, -2))) / 2.0
    trace_val = jnp.trace(force_ah, axis1=-2, axis2=-1)[..., None, None]
    drift_term = -(force_ah - (trace_val / 3.0) * jnp.eye(3, dtype=jnp.complex64))

    noise_raw = jax.random.normal(key, U_field.shape + (2,))
    noise_c = noise_raw[..., 0] + 1j * noise_raw[..., 1]
    noise_ah = (noise_c - jnp.conjugate(jnp.swapaxes(noise_c, -1, -2))) / 2.0
    trace_noise = jnp.trace(noise_ah, axis1=-2, axis2=-1)[..., None, None]
    noise_traceless = noise_ah - (trace_noise / 3.0) * jnp.eye(3, dtype=jnp.complex64)

    exponent = epsilon * drift_term + jnp.sqrt(epsilon) * noise_traceless
    update_matrix = jax.scipy.linalg.expm(exponent)
    return update_matrix @ U_field

# =============================================================================
# 2. ANALYSIS UTILITIES (Autocorrelation & Fitting)
# =============================================================================

def compute_integrated_autocorr(series):
    n = len(series)
    mean = np.mean(series)
    c0 = np.var(series)
    if c0 == 0: return 0.0
    tau = 0.5
    max_lag = n // 50
    for t in range(1, max_lag):
        ct = np.mean((series[:-t] - mean) * (series[t:] - mean))
        rho = ct / c0
        if rho <= 0: break
        tau += rho
    return tau

def fit_models(betas, taus, sigmas):
    # Safe fitting with error bars
    # Model A: Poly (Convexity)
    def model_poly(b, a, p): return a * (b**p)
    # Model B: Exp (Tunneling)
    def model_exp(b, a, c): return a * np.exp(c * np.sqrt(b))

    popt_poly, _ = curve_fit(model_poly, betas, taus, sigma=sigmas, absolute_sigma=True, p0=[0.1, 2.0], maxfev=10000)
    popt_exp, _ = curve_fit(model_exp, betas, taus, sigma=sigmas, absolute_sigma=True, p0=[0.01, 3.0], maxfev=10000)

    resid_poly = taus - model_poly(betas, *popt_poly)
    resid_exp  = taus - model_exp(betas, *popt_exp)

    chi2_poly = np.sum((resid_poly / sigmas)**2)
    chi2_exp  = np.sum((resid_exp / sigmas)**2)

    # AIC Calculation (k=2 parameters for both)
    AIC_poly = 2*2 + chi2_poly
    AIC_exp = 2*2 + chi2_exp

    return {"poly": AIC_poly, "exp": AIC_exp}

# =============================================================================
# 3. MAIN OPTIMIZED EXECUTION
# =============================================================================

def run_high_beta_optimized():
    print("Initializing T13 'High-Beta' (Optimized Settings)...")
    print("Target: Beta 5.8 -> 6.6 | Chains: 4 | Steps: 8,000")

    L = 8
    betas = [5.8, 6.0, 6.2, 6.4, 6.6]
    epsilon = 0.02

    # OPTIMIZED PARAMETERS:
    burn_in    = 1000
    run_length = 8000   # Reduced to ensure completion
    n_chains   = 4      # Maintain multi-chain for error bars

    master_key = jax.random.PRNGKey(2025)
    U0 = jnp.broadcast_to(jnp.eye(3, dtype=jnp.complex64), (L, L, L, L, 4, 3, 3))

    print("\n" + "=" * 90)
    print(f"{'Beta':<8} | {'<W> mean±std':<24} | {'tau mean±std':<24} | {'Samples (Total)'}")
    print("=" * 90)

    beta_list = []
    tau_means = []
    tau_stds  = []

    for i, beta in enumerate(betas):
        beta_key = jax.random.fold_in(master_key, i)
        chain_taus = []
        chain_ws = []

        for c in range(n_chains):
            chain_key = jax.random.fold_in(beta_key, c)
            U = U0

            # Fast Burn-in
            key = chain_key
            for _ in range(burn_in):
                key, sub = jax.random.split(key)
                U = langevin_step(U, sub, epsilon, beta)

            # Measurement Loop
            w_hist = []
            for _ in range(run_length):
                key, sub = jax.random.split(key)
                U = langevin_step(U, sub, epsilon, beta)
                w = wilson_loop_trace(U, (0, 0, 0, 0), 2, 2)
                w_hist.append(float(w))

            w_hist = np.asarray(w_hist)
            tau = compute_integrated_autocorr(w_hist)

            chain_ws.append(np.mean(w_hist))
            chain_taus.append(tau)

        chain_taus = np.array(chain_taus)
        chain_ws = np.array(chain_ws)

        # Stats
        t_mean = np.mean(chain_taus)
        t_std = np.std(chain_taus, ddof=1)
        w_mean = np.mean(chain_ws)
        w_std = np.std(chain_ws, ddof=1)

        beta_list.append(beta)
        tau_means.append(t_mean)
        tau_stds.append(t_std)

        print(f"{beta:<8.1f} | {w_mean:.4f} ± {w_std:.4f}   | {t_mean:.4f} ± {t_std:.4f}   | {n_chains*run_length}")

    print("=" * 90)

    # --- FINAL VERDICT ---
    print("\n>>> AIC MODEL COMPARISON <<< ")

    # Handle zero std for fitting stability
    sigmas = np.array(tau_stds)
    sigmas[sigmas == 0] = 0.1

    scores = fit_models(beta_list, tau_means, sigmas)
    print(f"Polynomial AIC (Convexity): {scores['poly']:.3f}")
    print(f"Exponential AIC (Tunneling): {scores['exp']:.3f}")

    diff = scores['exp'] - scores['poly']

    if diff > 2.0:
        print(f"\nWINNER: POLYNOMIAL (AIC better by {diff:.2f})")
        print("VERDICT: The Mass Gap mechanism is CONVEXITY.")
    elif diff < -2.0:
        print(f"\nWINNER: EXPONENTIAL (AIC better by {-diff:.2f})")
        print("VERDICT: The Mass Gap mechanism is TUNNELING.")
    else:
        print(f"\nRESULT: STATISTICAL TIE (Diff {diff:.2f}). Needs higher precision.")

if __name__ == "__main__":
    run_high_beta_optimized()


Initializing T13 'High-Beta' (Optimized Settings)...
Target: Beta 5.8 -> 6.6 | Chains: 4 | Steps: 8,000

Beta     | <W> mean±std             | tau mean±std             | Samples (Total)
5.8      | 0.1827 ± 0.0072   | 3.8181 ± 0.1649   | 32000
6.0      | 0.2008 ± 0.0069   | 3.7726 ± 0.2879   | 32000
6.2      | 0.2015 ± 0.0067   | 3.8427 ± 0.4234   | 32000
6.4      | 0.2142 ± 0.0067   | 3.8663 ± 0.0488   | 32000
6.6      | 0.2146 ± 0.0062   | 3.6820 ± 0.4288   | 32000

>>> AIC MODEL COMPARISON <<< 
Polynomial AIC (Convexity): 4.256
Exponential AIC (Tunneling): 4.256

RESULT: STATISTICAL TIE (Diff 0.00). Needs higher precision.
